# Healthcare Readmission Prediction
## 03 — Predictive Modeling and Algorithm Selection

**Objective:** Build and evaluate reproducible classification models for identifying hospital encounters associated with readmission within 30 days.

This notebook implements the Week 4 methodology using:
- Logistic Regression as an interpretable baseline
- Random Forest as a non-linear tree-based model
- Stratified train/test splitting
- Missing-value imputation and categorical encoding inside a `Pipeline`
- Accuracy, precision, recall, F1, ROC-AUC and PR-AUC
- Confusion matrices and ROC/precision-recall curves
- Threshold comparison
- Permutation importance for model interpretation

**Important:** Model results are generated from the dataset when the notebook is executed. They are not manually inserted.


## 1. Prediction problem

The original `readmitted` outcome contains three categories:

- `<30` = readmitted within 30 days
- `>30` = readmitted after 30 days
- `NO` = no recorded readmission

For binary classification:

**Target = 1:** `<30`  
**Target = 0:** `>30` or `NO`

The standard dataset contains 101,766 encounters, with 11,357 early readmissions (approximately 11.16%). Because the positive class is a minority, accuracy alone is not an adequate evaluation measure.

In [ ]:
# Install once in Google Colab if needed:
# %pip install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")


## 2. Load and prepare the dataset

In [ ]:
diabetes = fetch_ucirepo(id=296)

X_raw = diabetes.data.features.copy()
y_raw = diabetes.data.targets.copy()

df = pd.concat([X_raw, y_raw], axis=1)
df = df.replace("?", np.nan)

# Create binary target
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Positive cases:", int(df["readmitted_30d"].sum()))
print("Positive rate: {:.2f}%".format(df["readmitted_30d"].mean() * 100))


## 3. Feature selection and leakage control

The model uses variables that can reasonably be available around the index hospitalization.

Identifier fields such as `encounter_id` and `patient_nbr` are excluded because they identify records rather than describe patient risk.

The original `readmitted` column is also excluded because it directly contains the outcome used to create the target.

Potentially problematic post-outcome information is not used.

The selected variables include demographics, admission characteristics, prior utilization, hospital-stay information, laboratory/procedure counts, medications and diagnosis count.

In [ ]:
feature_columns = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "diabetesMed",
    "change",
    "insulin",
    "A1Cresult",
    "max_glu_serum"
]

# Keep only columns that exist in the loaded dataset
feature_columns = [c for c in feature_columns if c in df.columns]

X = df[feature_columns].copy()
y = df["readmitted_30d"].copy()

print("Number of model features:", len(feature_columns))
print("Features:")
print(feature_columns)


## 4. Train/test split

A **stratified 80/20 split** is used so that the proportion of early readmissions is approximately preserved in both datasets.

The test set remains untouched until final evaluation.

Preprocessing is fitted only on the training data through the scikit-learn `Pipeline`, reducing the risk of data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positive rate: {:.2f}%".format(y_train.mean() * 100))
print("Test positive rate: {:.2f}%".format(y_test.mean() * 100))


## 5. Numerical and categorical preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Model 1 — Logistic Regression

Logistic Regression is used as an interpretable baseline for binary classification.

The model estimates the probability of the positive class. `class_weight="balanced"` gives additional weight to the minority class during fitting, which is appropriate to investigate because only about 11.16% of encounters are early readmissions.

This does not guarantee better performance; the final decision is based on test-set metrics.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_probability = logistic_model.predict_proba(X_test)[:, 1]
logistic_prediction = (logistic_probability >= 0.50).astype(int)

print("Logistic Regression trained successfully.")


## 7. Model 2 — Random Forest

Random Forest is included because it can capture non-linear relationships and interactions that a simple linear model may miss.

It is also evaluated with the same train/test split and the same preprocessing approach so that the comparison is more consistent.

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_probability = random_forest_model.predict_proba(X_test)[:, 1]
rf_prediction = (rf_probability >= 0.50).astype(int)

print("Random Forest trained successfully.")


## 8. Evaluation metrics

For a binary healthcare classification problem:

- **Accuracy** = proportion of all predictions that are correct.
- **Precision** = proportion of predicted positives that are actually positive.
- **Recall** = proportion of actual positives detected by the model.
- **F1-score** = harmonic mean of precision and recall.
- **ROC-AUC** = ranking performance across classification thresholds.
- **PR-AUC** = area under the precision-recall curve and particularly informative when the positive class is relatively uncommon.

No single metric should be used alone.

In [ ]:
def evaluate_model(name, y_true, predictions, probabilities):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities)
    }

results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_prediction,
        logistic_probability
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        rf_prediction,
        rf_probability
    )
]).round(4)

display(results)


## 9. Confusion matrices

In [ ]:
logistic_cm = confusion_matrix(y_test, logistic_prediction)
rf_cm = confusion_matrix(y_test, rf_prediction)

print("Logistic Regression confusion matrix:")
display(pd.DataFrame(
    logistic_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("Random Forest confusion matrix:")
display(pd.DataFrame(
    rf_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))


### Confusion-matrix interpretation

The four cells are:

- **True Positive (TP):** early readmission correctly identified.
- **True Negative (TN):** non-early-readmission correctly identified.
- **False Positive (FP):** model flags an encounter that is not an early readmission.
- **False Negative (FN):** model misses an actual early readmission.

In a screening-oriented healthcare use case, false negatives may deserve particular attention because a missed high-risk encounter could mean that an opportunity for follow-up is not identified.

## 10. Classification reports

In [ ]:
print("Logistic Regression classification report:")
print(classification_report(
    y_test,
    logistic_prediction,
    target_names=["Not <30 days", "<30 days"],
    zero_division=0
))

print("Random Forest classification report:")
print(classification_report(
    y_test,
    rf_prediction,
    target_names=["Not <30 days", "<30 days"],
    zero_division=0
))


## 11. ROC curves

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, logistic_probability)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probability)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_log,
    tpr_log,
    label=f"Logistic Regression (AUC={roc_auc_score(y_test, logistic_probability):.3f})"
)
plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC={roc_auc_score(y_test, rf_probability):.3f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Precision-recall curves

In [ ]:
precision_log, recall_log, _ = precision_recall_curve(y_test, logistic_probability)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, rf_probability)

plt.figure(figsize=(8, 6))
plt.plot(
    recall_log,
    precision_log,
    label=f"Logistic Regression (PR-AUC={average_precision_score(y_test, logistic_probability):.3f})"
)
plt.plot(
    recall_rf,
    precision_rf,
    label=f"Random Forest (PR-AUC={average_precision_score(y_test, rf_probability):.3f})"
)
plt.title("Precision-Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Threshold analysis

A probability threshold of 0.50 is not automatically the best operational threshold.

Lowering the threshold usually increases recall while reducing precision; raising it can do the opposite.

The table below calculates the trade-off from the actual test predictions.

In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

threshold_rows = []

for threshold in thresholds:
    pred = (rf_probability >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })

threshold_table = pd.DataFrame(threshold_rows).round(3)
display(threshold_table)


### Threshold interpretation

Suppose a lower threshold increases recall substantially. That means more actual early-readmission cases are being detected, but more false positives may also be generated.

The appropriate threshold therefore depends on the intended workflow, available clinical resources, and the relative consequences of false negatives and false positives.

This notebook does **not** claim that one threshold is clinically optimal.

## 14. Permutation importance for model interpretation

In [ ]:
# Permutation importance is calculated on the original test features.
# It measures how much performance changes when a feature is randomly shuffled.

perm = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "mean_importance": perm.importances_mean,
    "std_importance": perm.importances_std
}).sort_values("mean_importance", ascending=False)

display(importance)


## 15. Model comparison summary

In [ ]:
comparison = results.sort_values(
    by=["pr_auc", "recall", "f1"],
    ascending=False
).reset_index(drop=True)

display(comparison)


## 16. Model-selection rule

The model should not be selected simply because it has the highest accuracy.

For this imbalanced healthcare problem, the selection process should consider:

1. PR-AUC
2. Recall
3. Precision
4. F1-score
5. ROC-AUC
6. Calibration and threshold behaviour
7. Interpretability
8. Operational feasibility

The final choice should be justified using the actual test-set results printed above.

## 17. Worked metric example

For illustration only, if a model produced:

- TP = 150
- TN = 1,500
- FP = 200
- FN = 150

then:

**Accuracy**

(150 + 1,500) / 2,000 = **82.5%**

**Precision**

150 / (150 + 200) = **42.9%**

**Recall**

150 / (150 + 150) = **50.0%**

**F1-score**

2 × (0.429 × 0.500) / (0.429 + 0.500) ≈ **46.2%**

These numbers are an instructional example, not the model result. The actual results for this project are generated by the evaluation cells above.

## 18. Healthcare interpretation and limitations

The model is intended as a **decision-support research prototype**, not a clinical decision-maker.

Important limitations include:

- historical data from 1999–2008 may not represent current clinical practice;
- class imbalance affects evaluation;
- missing information may contain systematic patterns;
- hospital-specific practices may influence the data;
- associations do not establish causation;
- predictive performance does not automatically demonstrate clinical usefulness;
- fairness should be checked across demographic subgroups before deployment;
- external validation and prospective clinical evaluation would be required before real-world use.

### Final conclusion

This notebook establishes a reproducible predictive-modelling workflow. Logistic Regression provides an interpretable baseline, while Random Forest provides a non-linear comparison. The models are evaluated with metrics that are more informative than accuracy alone for a minority-class readmission outcome. Threshold analysis and permutation importance add further evidence for operational and interpretability decisions.